# Notebook 03: Embedding Model Experiment — All-in-One (Google Colab)

## MedQA Thesis — Systematic Comparison of RAG Architectures

This is a **single-cell experiment runner** that tests **4 Embedding Models** across all 4 RAG architectures.  
VectorDB is fixed to **FAISS** (best performer from Notebook 02a).

| Component | Value |
|-----------|-------|
| **Chunking** | Semantic Passage (200 tokens) — FIXED |
| **VectorDB** | FAISS — FIXED |
| **LLM** | LLaMA 3.3 70B via Groq — FIXED |
| **Embeddings** | all-MiniLM-L6-v2, bge-base-en-v1.5, bge-large-en-v1.5, MedCPT — **VARIABLE** |
| **RAG Architectures** | Naive, Sparse, Hybrid, Multi-Hop |
| **Sample Size** | 2 questions (quick validation) |

In [ ]:
# ==============================================================
# GOOGLE COLAB SETUP — Mount Drive & Install Packages
# ==============================================================
from google.colab import drive
drive.mount('/content/drive')

COLAB_PROJECT_ROOT = '/content/drive/MyDrive/MedQA-Thesis'

import subprocess, sys
packages = [
    "langchain==0.3.25", "langchain-core==0.3.83", "langchain-community==0.3.25",
    "langchain-text-splitters==0.3.11",
    "sentence-transformers", "faiss-cpu", "rank-bm25", "groq",
    "openpyxl", "rouge-score", "bert-score",
    "pandas", "numpy", "matplotlib", "seaborn", "tqdm",
    "requests==2.32.4",
]
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q"] + packages,
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

import os
processed_path = os.path.join(COLAB_PROJECT_ROOT, 'data', 'processed')
for f in ["medqa_dev.parquet", "textbook_corpus.json"]:
    if not os.path.exists(os.path.join(processed_path, f)):
        raise FileNotFoundError(f"Missing: {processed_path}/{f} — Run Notebook 01 first.")

print("Setup complete. All packages installed. Data verified on Drive.")

In [ ]:
# ==============================================================
# 03: EMBEDDING MODEL EXPERIMENT — ALL-IN-ONE CELL
# ==============================================================
# This cell runs the entire Embedding comparison experiment:
#   1.  Setup & imports
#   2.  Paths & config (4 embedding models)
#   3.  Load data
#   4.  Chunk textbooks
#   5.  Embedding model wrapper (unified interface)
#   6.  Build BM25 index (shared across all embeddings)
#   7.  Evaluation metrics (20 metrics)
#   8.  Prompt template & LLM call
#   9.  Retrieval functions (FAISS + BM25)
#   10. RAG architectures (Naive, Sparse, Hybrid, Multi-Hop)
#   11. Excel writer
#   12. Master experiment runner
#   13. Run all experiments (4 embeddings × 4 RAG types)
#   14. Comparison & visualization
# ==============================================================

# ============================================================
# SECTION 1: IMPORTS
# ============================================================
import json, os, re, time, gc, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import openpyxl
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModel
from langchain_text_splitters import RecursiveCharacterTextSplitter
import faiss
from rank_bm25 import BM25Okapi

from groq import Groq
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn

warnings.filterwarnings("ignore")

# ============================================================
# SECTION 2: PATHS & CONFIG
# ============================================================
PROJECT_ROOT   = Path(COLAB_PROJECT_ROOT)
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_INDICES   = PROJECT_ROOT / "data" / "indices"
EXPERIMENTS    = PROJECT_ROOT / "experiments"
RESULTS_DIR    = PROJECT_ROOT / "results" / "experiments"
EXCEL_PATH     = EXPERIMENTS / "ThesisExperiment.xlsx"

for d in [DATA_INDICES, RESULTS_DIR, RESULTS_DIR / "plots"]:
    d.mkdir(parents=True, exist_ok=True)

# --- API Key ---
GROQ_API_KEY = "your-groq-api-key-here"         # <-- Paste your Groq key
groq_client = Groq(api_key=GROQ_API_KEY)

# --- Fixed experiment configuration ---
CHUNKING_NAME        = "Semantic Passage (200 tokens)"
CHUNK_MAX_TOKENS     = 200
CHUNK_OVERLAP        = 20
VECTORDB_NAME        = "FAISS"
LLM_MODEL            = "llama-3.3-70b-versatile"
LLM_TEMPERATURE      = 0
TOP_K                = 5
MAX_HOPS             = 3
RRF_K                = 60
FAITHFULNESS_THRESHOLD = 0.5
MAX_CHUNK_CHARS      = 1200
SAMPLE_SIZE          = 2  # Quick validation — increase for full run

# --- 4 Embedding models to compare ---
EMBEDDING_CONFIGS = {
    "all-MiniLM-L6-v2": {
        "model_name": "all-MiniLM-L6-v2",
        "dim": 384,
        "type": "sbert",
        "query_prefix": "",
    },
    "bge-base-en-v1.5": {
        "model_name": "BAAI/bge-base-en-v1.5",
        "dim": 768,
        "type": "sbert",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
    },
    "bge-large-en-v1.5": {
        "model_name": "BAAI/bge-large-en-v1.5",
        "dim": 1024,
        "type": "sbert",
        "query_prefix": "Represent this sentence for searching relevant passages: ",
    },
    "MedCPT": {
        "query_model": "ncbi/MedCPT-Query-Encoder",
        "article_model": "ncbi/MedCPT-Article-Encoder",
        "dim": 768,
        "type": "medcpt",
        "query_prefix": "",
    },
}

# Excel row mapping: (rag_type, embedding_name) -> row number
EXCEL_ROW_MAP = {
    ("naive",    "all-MiniLM-L6-v2"):  2,  ("naive",    "bge-base-en-v1.5"):  3,
    ("naive",    "bge-large-en-v1.5"): 4,  ("naive",    "MedCPT"):             5,
    ("sparse",   "all-MiniLM-L6-v2"):  7,  ("sparse",   "bge-base-en-v1.5"):  8,
    ("sparse",   "bge-large-en-v1.5"): 9,  ("sparse",   "MedCPT"):             10,
    ("hybrid",   "all-MiniLM-L6-v2"):  12, ("hybrid",   "bge-base-en-v1.5"):  13,
    ("hybrid",   "bge-large-en-v1.5"): 14, ("hybrid",   "MedCPT"):             15,
    ("multihop", "all-MiniLM-L6-v2"):  17, ("multihop", "bge-base-en-v1.5"):  18,
    ("multihop", "bge-large-en-v1.5"): 19, ("multihop", "MedCPT"):             20,
}

print(f"Config: {CHUNKING_NAME} | VectorDB: {VECTORDB_NAME} | LLM: {LLM_MODEL}")
print(f"Sample: {SAMPLE_SIZE} questions | Embeddings: {list(EMBEDDING_CONFIGS.keys())}")

# ============================================================
# SECTION 3: LOAD DATA
# ============================================================
df_dev = pd.read_parquet(DATA_PROCESSED / "medqa_dev.parquet")
df_dev["options"] = df_dev.apply(
    lambda row: {"A": row["option_a"], "B": row["option_b"],
                 "C": row["option_c"], "D": row["option_d"]}, axis=1)

with open(DATA_PROCESSED / "textbook_corpus.json", "r") as f:
    textbook_corpus = json.load(f)

df_sample = df_dev.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Loaded {len(df_dev)} dev questions, sampled {SAMPLE_SIZE}")

# ============================================================
# SECTION 4: CHUNK TEXTBOOKS
# ============================================================
def semantic_passage_chunk(text, max_tokens=200, overlap=20):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_tokens * 4, chunk_overlap=overlap,
        separators=["\n\n\n", "\n\n", "\n", ". ", "; ", ", ", " "],
        length_function=lambda x: len(x.split()), is_separator_regex=False)
    chunks = splitter.split_text(text)
    return [c.strip() for c in chunks if len(c.split()) >= 10]

print("Chunking textbooks...")
all_chunks = []
chunk_metadata = []
for book in tqdm(textbook_corpus, desc="Chunking"):
    for i, ct in enumerate(semantic_passage_chunk(book["text"], CHUNK_MAX_TOKENS, CHUNK_OVERLAP)):
        all_chunks.append(ct)
        chunk_metadata.append({"book_name": book["book_name"], "chunk_id": f"{book['book_name']}_chunk_{i}", "text": ct})
print(f"Total chunks: {len(all_chunks):,} | Avg: {np.mean([len(c.split()) for c in all_chunks]):.0f} words")

# ============================================================
# SECTION 5: EMBEDDING MODEL WRAPPER
# ============================================================
class EmbeddingWrapper:
    """Unified interface for SentenceTransformer and MedCPT models."""

    def __init__(self, config, emb_name):
        self.config = config
        self.emb_name = emb_name
        self.model_type = config["type"]
        self.dim = config["dim"]
        self.query_prefix = config.get("query_prefix", "")

        if self.model_type == "sbert":
            print(f"  Loading SentenceTransformer: {config['model_name']}")
            self.model = SentenceTransformer(config["model_name"])
            self.query_model = None
        elif self.model_type == "medcpt":
            print(f"  Loading MedCPT article encoder: {config['article_model']}")
            self.article_tokenizer = AutoTokenizer.from_pretrained(config["article_model"])
            self.article_model = AutoModel.from_pretrained(config["article_model"])
            self.article_model.eval()
            print(f"  Loading MedCPT query encoder: {config['query_model']}")
            self.query_tokenizer = AutoTokenizer.from_pretrained(config["query_model"])
            self.query_model = AutoModel.from_pretrained(config["query_model"])
            self.query_model.eval()

    def encode_documents(self, texts, batch_size=256, show_progress_bar=True):
        """Encode document/chunk texts."""
        if self.model_type == "sbert":
            return self.model.encode(texts, show_progress_bar=show_progress_bar,
                                    batch_size=batch_size, normalize_embeddings=True)
        elif self.model_type == "medcpt":
            return self._encode_medcpt(texts, self.article_tokenizer,
                                      self.article_model, batch_size, show_progress_bar)

    def encode_query(self, query_text):
        """Encode a single query."""
        if self.model_type == "sbert":
            q = self.query_prefix + query_text
            return self.model.encode(q, normalize_embeddings=True)
        elif self.model_type == "medcpt":
            emb = self._encode_medcpt([query_text], self.query_tokenizer,
                                      self.query_model, batch_size=1,
                                      show_progress_bar=False)
            return emb[0]

    def _encode_medcpt(self, texts, tokenizer, model, batch_size=64,
                       show_progress_bar=False):
        all_embeds = []
        iterator = range(0, len(texts), batch_size)
        if show_progress_bar:
            iterator = tqdm(iterator, desc="MedCPT encoding")
        for i in iterator:
            batch = texts[i:i + batch_size]
            encoded = tokenizer(
                batch, truncation=True, padding=True,
                return_tensors="pt", max_length=512
            )
            with torch.no_grad():
                embeds = model(**encoded).last_hidden_state[:, 0, :]
            # L2-normalise for cosine similarity via inner product
            embeds = torch.nn.functional.normalize(embeds, p=2, dim=1)
            all_embeds.append(embeds.cpu().numpy())
        return np.concatenate(all_embeds, axis=0).astype(np.float32)

    def cleanup(self):
        """Free GPU/CPU memory."""
        if self.model_type == "sbert":
            del self.model
        elif self.model_type == "medcpt":
            del self.article_model, self.article_tokenizer
            del self.query_model, self.query_tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print("EmbeddingWrapper defined.")

# ============================================================
# SECTION 6: BUILD BM25 INDEX (shared across all embeddings)
# ============================================================
print("Building BM25 index...")
tokenised_chunks = [chunk.lower().split() for chunk in all_chunks]
bm25_index = BM25Okapi(tokenised_chunks)
print(f"BM25 index: {len(tokenised_chunks):,} chunks")

# ============================================================
# SECTION 7: EVALUATION METRICS
# ============================================================
def parse_answer_letter(response_text):
    if not response_text: return None
    m = re.search(r"[Aa]nswer\s*[:=]\s*\(?([A-Da-d])\)?", response_text)
    if m: return m.group(1).upper()
    m = re.search(r"^\s*\(?([A-Da-d])\)?[\s\)\.]", response_text, re.MULTILINE)
    if m: return m.group(1).upper()
    m = re.search(r"\b([A-Da-d])\b", response_text)
    if m: return m.group(1).upper()
    return None

def compute_classification_metrics(predictions, ground_truths):
    correct = sum(1 for p, g in zip(predictions, ground_truths) if p == g)
    accuracy = correct / len(predictions) if predictions else 0
    labels = ["A", "B", "C", "D"]
    precisions, recalls = [], []
    for label in labels:
        tp = sum(1 for p, g in zip(predictions, ground_truths) if p == label and g == label)
        fp = sum(1 for p, g in zip(predictions, ground_truths) if p == label and g != label)
        fn = sum(1 for p, g in zip(predictions, ground_truths) if p != label and g == label)
        precisions.append(tp / (tp + fp) if (tp + fp) > 0 else 0)
        recalls.append(tp / (tp + fn) if (tp + fn) > 0 else 0)
    mp, mr = np.mean(precisions), np.mean(recalls)
    mf1 = 2 * mp * mr / (mp + mr) if (mp + mr) > 0 else 0
    return {"Accuracy": round(accuracy, 4), "Precision": round(mp, 4), "Recall": round(mr, 4), "F1": round(mf1, 4)}

def compute_token_f1(pred, ref):
    pt, rt = pred.lower().split(), ref.lower().split()
    common = sum((Counter(pt) & Counter(rt)).values())
    if common == 0: return 0.0
    p, r = common / len(pt) if pt else 0, common / len(rt) if rt else 0
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

def compute_exact_match(pred, ref):
    return 1.0 if pred.strip().upper() == ref.strip().upper() else 0.0

rouge_scorer_obj = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)
def compute_rouge(pred, ref):
    s = rouge_scorer_obj.score(ref, pred)
    return {"ROUGE_1": round(s["rouge1"].fmeasure, 4), "ROUGE_L": round(s["rougeL"].fmeasure, 4)}

def compute_retrieval_metrics(retrieved_list, df_q, chunk_texts):
    recall_at = {3: [], 5: [], 10: []}
    mrr_scores = []
    for i, indices in enumerate(retrieved_list):
        row = df_q.iloc[i]
        ans = row["answer"].lower() if "answer" in row else ""
        terms = [w for w in ans.split() if len(w) > 3] or ans.split()
        found_rank = None
        for rank, idx in enumerate(indices):
            if idx < len(chunk_texts) and any(t in chunk_texts[idx].lower() for t in terms):
                if found_rank is None: found_rank = rank + 1
        for k in [3, 5, 10]:
            hit = any(idx < len(chunk_texts) and any(t in chunk_texts[idx].lower() for t in terms) for idx in indices[:k])
            recall_at[k].append(int(hit))
        mrr_scores.append(1.0 / found_rank if found_rank else 0.0)
    return {"Recall@3": round(np.mean(recall_at[3]), 4), "Recall@5": round(np.mean(recall_at[5]), 4),
            "Recall@10": round(np.mean(recall_at[10]), 4), "MRR": round(np.mean(mrr_scores), 4)}

# --- RAGAS metrics (LLM-based) ---
def ragas_faithfulness_single(answer_text, context_text, client, model=LLM_MODEL):
    if not answer_text or not context_text: return 0.0
    try:
        resp = client.chat.completions.create(model=model, temperature=0, max_tokens=500,
            messages=[{"role": "user", "content": f"Extract all factual claims from this answer. Number each. If none, say NO_CLAIMS.\n\nAnswer: {answer_text}\n\nClaims:"}])
        claims_text = resp.choices[0].message.content.strip()
        if "NO_CLAIMS" in claims_text: return 1.0
        claims = [l.strip() for l in claims_text.split("\n") if l.strip() and l.strip()[0].isdigit()]
        if not claims: return 1.0
        supported = 0
        for claim in claims:
            r2 = client.chat.completions.create(model=model, temperature=0, max_tokens=20,
                messages=[{"role": "user", "content": f'Context: {context_text[:3000]}\n\nClaim: {claim}\n\nIs this SUPPORTED or NOT_SUPPORTED? Answer one word:'}])
            v = r2.choices[0].message.content.strip().upper()
            if "SUPPORTED" in v and "NOT" not in v: supported += 1
            time.sleep(0.5)
        return supported / len(claims)
    except Exception as e:
        print(f"    Faithfulness error: {e}"); return np.nan

def ragas_answer_relevance_single(question, answer_text, client, model=LLM_MODEL):
    if not answer_text: return 0.0
    try:
        r = client.chat.completions.create(model=model, temperature=0, max_tokens=10,
            messages=[{"role": "user", "content": f"Rate relevance 0.0-1.0. Return ONLY a number.\n\nQuestion: {question}\nAnswer: {answer_text}\n\nScore:"}])
        return min(max(float(re.search(r"[0-9]*\.?[0-9]+", r.choices[0].message.content).group()), 0), 1)
    except: return np.nan

def ragas_context_precision_single(question, ctx_chunks, answer_text, client, model=LLM_MODEL):
    if not ctx_chunks: return 0.0
    try:
        passages = "\n".join(f"Passage {i+1}: {c[:300]}" for i, c in enumerate(ctx_chunks[:5]))
        r = client.chat.completions.create(model=model, temperature=0, max_tokens=50,
            messages=[{"role": "user", "content": f"For each passage, is it relevant? Return comma-separated 1/0.\n\nQuestion: {question}\n\n{passages}\n\nRelevance:"}])
        relevance = [int(n) for n in re.findall(r"[01]", r.choices[0].message.content)[:len(ctx_chunks)]]
        if not any(relevance): return 0.0
        precs, running = [], 0
        for k, rel in enumerate(relevance):
            if rel: running += 1; precs.append(running / (k + 1))
        return np.mean(precs) if precs else 0.0
    except: return np.nan

def ragas_context_recall_single(question, context_text, answer_text, client, model=LLM_MODEL):
    if not context_text: return 0.0
    try:
        r = client.chat.completions.create(model=model, temperature=0, max_tokens=10,
            messages=[{"role": "user", "content": f"Does context contain enough info for the answer? Rate 0.0-1.0. ONLY a number.\n\nQuestion: {question}\nAnswer: {answer_text}\nContext: {context_text[:3000]}\n\nScore:"}])
        return min(max(float(re.search(r"[0-9]*\.?[0-9]+", r.choices[0].message.content).group()), 0), 1)
    except: return np.nan

print("All metrics defined.")

# ============================================================
# SECTION 8: PROMPT TEMPLATE & LLM CALL
# ============================================================
PROMPT_TEMPLATE = (
    "You are a medical expert answering USMLE-style clinical questions.\n"
    "Use ONLY the provided evidence passages to answer. If the evidence\n"
    "is insufficient, state that rather than guessing.\n\n"
    "Evidence Passages:\n{context}\n\n"
    "Question: {question}\n\n"
    "Options:\n"
    "A) {option_a}\nB) {option_b}\nC) {option_c}\nD) {option_d}\n\n"
    "Instructions:\n"
    "1. Analyze the evidence passages relevant to the question.\n"
    "2. Select the single best answer (A, B, C, or D).\n"
    "3. Provide a brief explanation citing specific evidence passages.\n\n"
    "Answer format:\nAnswer: [letter]\nExplanation: [your reasoning with evidence citations]\n"
)

def build_prompt(question, options, context_chunks):
    truncated = [c[:MAX_CHUNK_CHARS] for c in context_chunks]
    context_str = "\n\n".join(f"[Passage {i+1}]: {c}" for i, c in enumerate(truncated))
    ov = list(options.values()) if isinstance(options, dict) else options
    return PROMPT_TEMPLATE.format(context=context_str, question=question,
        option_a=ov[0] if len(ov)>0 else "", option_b=ov[1] if len(ov)>1 else "",
        option_c=ov[2] if len(ov)>2 else "", option_d=ov[3] if len(ov)>3 else "")

def call_llm(prompt, client=None, model=None, temperature=0, max_tokens=500):
    if client is None: client = groq_client
    if model is None: model = LLM_MODEL
    try:
        r = client.chat.completions.create(model=model, messages=[{"role":"user","content":prompt}], temperature=temperature, max_tokens=max_tokens)
        return r.choices[0].message.content.strip()
    except Exception as e:
        print(f"  LLM Error: {e}"); time.sleep(2); return ""

# ============================================================
# SECTION 9: RETRIEVAL FUNCTIONS (FAISS + BM25)
# ============================================================
def faiss_retrieve(query_emb, index, top_k=5):
    scores, indices = index.search(np.array([query_emb], dtype=np.float32), top_k)
    return indices[0].tolist(), scores[0].tolist()

def bm25_retrieve(query_text, bm25_idx, top_k=5):
    scores = bm25_idx.get_scores(query_text.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return top_idx.tolist(), scores[top_idx].tolist()

def rrf_fuse(ranking_lists, k=60):
    fused = {}
    for ranking in ranking_lists:
        for rank, (idx, _) in enumerate(ranking):
            fused[idx] = fused.get(idx, 0) + 1.0 / (k + rank + 1)
    return sorted(fused.items(), key=lambda x: x[1], reverse=True)

print("Retrieval functions defined.")

# ============================================================
# SECTION 10: RAG ARCHITECTURES
# ============================================================
def run_naive_rag(question, options, emb_wrapper, faiss_idx, top_k=5):
    query_emb = emb_wrapper.encode_query(question)
    indices, scores = faiss_retrieve(query_emb, faiss_idx, top_k)
    ctx = [all_chunks[i] for i in indices if i < len(all_chunks)]
    resp = call_llm(build_prompt(question, options, ctx))
    return parse_answer_letter(resp), resp, indices, ctx

def run_sparse_rag(question, options, bm25_idx, top_k=5):
    indices, scores = bm25_retrieve(question, bm25_idx, top_k)
    ctx = [all_chunks[i] for i in indices if i < len(all_chunks)]
    resp = call_llm(build_prompt(question, options, ctx))
    return parse_answer_letter(resp), resp, indices, ctx

def run_hybrid_rag(question, options, emb_wrapper, bm25_idx, faiss_idx,
                   top_k=5, rrf_k=60):
    query_emb = emb_wrapper.encode_query(question)
    fetch_k = top_k * 3
    di, ds = faiss_retrieve(query_emb, faiss_idx, fetch_k)
    si, ss = bm25_retrieve(question, bm25_idx, fetch_k)
    fused = rrf_fuse([list(zip(di, ds)), list(zip(si, ss))], k=rrf_k)
    top_idx = [idx for idx, _ in fused[:top_k]]
    ctx = [all_chunks[i] for i in top_idx if i < len(all_chunks)]
    resp = call_llm(build_prompt(question, options, ctx))
    return parse_answer_letter(resp), resp, top_idx, ctx

def run_multihop_rag(question, options, emb_wrapper, faiss_idx,
                     top_k=5, max_hops=3):
    chain, all_ret_idx, all_ctx = [], [], []
    # Decompose question
    dresp = call_llm(
        f"Break this medical question into 2-3 sub-questions. Number each.\n\n"
        f"Question: {question}\n\nSub-questions:", max_tokens=200)
    subs = [l.strip() for l in dresp.split("\n")
            if l.strip() and (l.strip()[0].isdigit() or l.strip().startswith("-"))]
    if not subs: subs = [question]
    chain.append({"step": "decomposition", "sub_questions": subs})
    # Hop 1: retrieve for each sub-question
    for sq in subs:
        qe = emb_wrapper.encode_query(sq)
        idx, sc = faiss_retrieve(qe, faiss_idx, top_k)
        all_ret_idx.extend(idx)
        all_ctx.extend([all_chunks[i] for i in idx if i < len(all_chunks)])
    chain.append({"step": "hop_1", "num_chunks": len(all_ctx)})
    # Hop 2+: iterative refinement
    for hop in range(2, max_hops + 1):
        evidence_str = " ".join(all_ctx[-10:])[:2000]
        gap_resp = call_llm(
            f'Question: {question}\nEvidence: {evidence_str}\n\n'
            f'Is evidence SUFFICIENT? If not, give 1-2 refined queries:',
            max_tokens=200)
        time.sleep(0.5)
        if "SUFFICIENT" in gap_resp.upper():
            chain.append({"step": f"hop_{hop}", "status": "sufficient"}); break
        rqs = [l.strip() for l in gap_resp.split("\n")
               if l.strip() and len(l.strip()) > 10][:2]
        if not rqs: break
        for rq in rqs:
            qe = emb_wrapper.encode_query(rq)
            idx, sc = faiss_retrieve(qe, faiss_idx, top_k)
            all_ret_idx.extend(idx)
            all_ctx.extend([all_chunks[i] for i in idx if i < len(all_chunks)])
        chain.append({"step": f"hop_{hop}", "queries": rqs})
    # Deduplicate evidence
    seen, ui, uc = set(), [], []
    for idx, c in zip(all_ret_idx, all_ctx):
        if idx not in seen: seen.add(idx); ui.append(idx); uc.append(c)
    fi, fc = ui[:top_k*2], uc[:top_k*2]
    resp = call_llm(build_prompt(question, options, fc))
    return parse_answer_letter(resp), resp, fi, fc, chain

print("All 4 RAG architectures defined.")

# ============================================================
# SECTION 11: EXCEL WRITER
# ============================================================
EXCEL_COL_MAP = {
    "A":"SN","B":"RAG_Name","C":"Chunking_Technique","D":"Embedding","E":"VectorDB","F":"Generator_Model",
    "G":"Accuracy","H":"Precision","I":"Recall","J":"F1","K":"Token_F1","L":"Exact_Match",
    "M":"ROUGE_1","N":"ROUGE_L","O":"BERTScore_P","P":"BERTScore_R","Q":"BERTScore_F1",
    "R":"Recall@3","S":"Recall@5","T":"Recall@10","U":"MRR",
    "V":"RAGAS_Faithfulness","W":"RAGAS_Hallucination_Rate","X":"RAGAS_Answer_Relevance",
    "Y":"RAGAS_Context_Precision","Z":"RAGAS_Context_Recall"}
METRIC_TO_COL = {v: k for k, v in EXCEL_COL_MAP.items()}

def write_to_excel(excel_path, row, config, metrics):
    if not excel_path.exists():
        print(f"  Excel not found, skipping write"); return
    wb = openpyxl.load_workbook(excel_path)
    ws = wb["Embedding Comparision"]
    for key, val in {**config, **metrics}.items():
        col = METRIC_TO_COL.get(key)
        if col and val is not None:
            ws[f"{col}{row}"] = round(val, 4) if isinstance(val, float) else val
    wb.save(excel_path)
    print(f"  Excel row {row} updated")

# ============================================================
# SECTION 12: MASTER EXPERIMENT RUNNER
# ============================================================
all_results = {}

def run_experiment(rag_type, emb_name, df_q, emb_wrapper, faiss_idx):
    rag_display = {"naive":"Naive RAG","sparse":"Sparse RAG",
                   "hybrid":"Hybrid RAG","multihop":"Multi-Hop RAG"}[rag_type]
    excel_row = EXCEL_ROW_MAP.get((rag_type, emb_name))
    print(f"\n{'='*60}\nEXPERIMENT: {rag_display} + {emb_name}")
    print(f"Excel Row: {excel_row} | Questions: {len(df_q)}\n{'='*60}")

    preds, gts, resps = [], [], []
    all_ret, all_ctx_t, correct_txts, q_txts = [], [], [], []

    for i in tqdm(range(len(df_q)), desc=f"{rag_display}+{emb_name}"):
        row = df_q.iloc[i]
        q, opts = row["question"], row["options"]
        gt = row.get("answer_idx", row.get("answer", "A"))
        ov = list(opts.values()) if isinstance(opts, dict) else opts
        ct = ov[{"A":0,"B":1,"C":2,"D":3}.get(gt, 0)]

        if rag_type == "naive":
            p, r, ri, rc = run_naive_rag(q, opts, emb_wrapper, faiss_idx, TOP_K)
        elif rag_type == "sparse":
            p, r, ri, rc = run_sparse_rag(q, opts, bm25_index, TOP_K)
        elif rag_type == "hybrid":
            p, r, ri, rc = run_hybrid_rag(q, opts, emb_wrapper, bm25_index,
                                          faiss_idx, TOP_K, RRF_K)
        elif rag_type == "multihop":
            p, r, ri, rc, _ = run_multihop_rag(q, opts, emb_wrapper, faiss_idx,
                                                TOP_K, MAX_HOPS)

        preds.append(p or "X"); gts.append(gt); resps.append(r)
        all_ret.append(ri); all_ctx_t.append(rc)
        correct_txts.append(ct); q_txts.append(q)
        time.sleep(0.5)

    # --- Compute all 20 metrics ---
    print("Computing metrics...")
    cls = compute_classification_metrics(preds, gts)
    tf1 = round(np.mean([compute_token_f1(p, g) for p, g in zip(preds, gts)]), 4)
    em = round(np.mean([compute_exact_match(p, g) for p, g in zip(preds, gts)]), 4)

    r1s, rLs = [], []
    for resp, ct in zip(resps, correct_txts):
        rg = compute_rouge(resp, ct) if resp and ct else {"ROUGE_1": 0, "ROUGE_L": 0}
        r1s.append(rg["ROUGE_1"]); rLs.append(rg["ROUGE_L"])

    try:
        P, R, F = bert_score_fn(resps, correct_txts, lang="en", verbose=False,
                                rescale_with_baseline=True)
        bsp = round(P.mean().item(), 4)
        bsr = round(R.mean().item(), 4)
        bsf = round(F.mean().item(), 4)
    except: bsp = bsr = bsf = np.nan

    ret = compute_retrieval_metrics(all_ret, df_q, all_chunks)

    # RAGAS metrics (LLM-based)
    print("Computing RAGAS metrics...")
    fs, rs, cps, crs = [], [], [], []
    for i in tqdm(range(len(df_q)), desc="RAGAS", leave=False):
        ctx_str = "\n".join(all_ctx_t[i]) if all_ctx_t[i] else ""
        fs.append(ragas_faithfulness_single(resps[i], ctx_str, groq_client))
        rs.append(ragas_answer_relevance_single(q_txts[i], resps[i], groq_client))
        cps.append(ragas_context_precision_single(q_txts[i], all_ctx_t[i],
                                                  correct_txts[i], groq_client))
        crs.append(ragas_context_recall_single(q_txts[i], ctx_str,
                                               correct_txts[i], groq_client))
        time.sleep(0.3)

    vf = [s for s in fs if not np.isnan(s)]
    hall_rate = round(sum(1 for s in vf if s < FAITHFULNESS_THRESHOLD) / len(vf), 4) if vf else np.nan

    metrics = {
        **cls, "Token_F1": tf1, "Exact_Match": em,
        "ROUGE_1": round(np.mean(r1s), 4), "ROUGE_L": round(np.mean(rLs), 4),
        "BERTScore_P": bsp, "BERTScore_R": bsr, "BERTScore_F1": bsf, **ret,
        "RAGAS_Faithfulness": round(np.nanmean(fs), 4),
        "RAGAS_Hallucination_Rate": hall_rate,
        "RAGAS_Answer_Relevance": round(np.nanmean(rs), 4),
        "RAGAS_Context_Precision": round(np.nanmean(cps), 4),
        "RAGAS_Context_Recall": round(np.nanmean(crs), 4),
    }

    config = {
        "SN": excel_row, "RAG_Name": rag_display,
        "Chunking_Technique": CHUNKING_NAME,
        "Embedding": emb_name, "VectorDB": VECTORDB_NAME,
        "Generator_Model": LLM_MODEL,
    }

    if excel_row:
        write_to_excel(EXCEL_PATH, excel_row, config, metrics)
    all_results[f"{rag_type}_{emb_name}"] = {"config": config, "metrics": metrics}
    print(f"\n--- {rag_display} + {emb_name} ---")
    for k, v in metrics.items(): print(f"  {k:30s}: {v}")
    return metrics

# ============================================================
# SECTION 13: RUN ALL EXPERIMENTS
# ============================================================
rag_types = ["naive", "sparse", "hybrid", "multihop"]

for emb_name, emb_config in EMBEDDING_CONFIGS.items():
    print(f"\n{'#'*60}")
    print(f"# PHASE: {emb_name} (dim={emb_config['dim']})")
    print(f"{'#'*60}")

    # --- Load embedding model ---
    emb_wrapper = EmbeddingWrapper(emb_config, emb_name)

    # --- Generate chunk embeddings ---
    print(f"Embedding {len(all_chunks):,} chunks with {emb_name}...")
    chunk_embeddings = emb_wrapper.encode_documents(all_chunks, batch_size=256,
                                                    show_progress_bar=True)
    chunk_embeddings = np.array(chunk_embeddings, dtype=np.float32)
    print(f"Embeddings shape: {chunk_embeddings.shape}")

    # --- Build FAISS index ---
    print(f"Building FAISS index (dim={emb_config['dim']})...")
    faiss_index = faiss.IndexFlatIP(emb_config["dim"])
    faiss_index.add(chunk_embeddings)
    print(f"FAISS: {faiss_index.ntotal:,} vectors")

    # --- Run 4 RAG experiments ---
    for rag in rag_types:
        run_experiment(rag, emb_name, df_sample, emb_wrapper, faiss_index)

    # --- Cleanup to free memory before next model ---
    print(f"\nCleaning up {emb_name}...")
    emb_wrapper.cleanup()
    del chunk_embeddings, faiss_index, emb_wrapper
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"{emb_name} phase complete.\n")

# ============================================================
# SECTION 14: COMPARISON & VISUALIZATION
# ============================================================
if all_results:
    comp_data = []
    for key, res in all_results.items():
        parts = key.split("_", 1)
        rag_d = {"naive":"Naive RAG","sparse":"Sparse RAG",
                 "hybrid":"Hybrid RAG","multihop":"Multi-Hop RAG"}.get(parts[0], parts[0])
        emb_d = parts[1]
        comp_data.append({"RAG": rag_d, "Embedding": emb_d, **res["metrics"]})
    comp_df = pd.DataFrame(comp_data)

    print("\n" + "="*80 + "\nFULL COMPARISON TABLE\n" + "="*80)
    print(comp_df.to_string(index=False))

    comp_df.to_csv(RESULTS_DIR / "03_embedding_comparison.csv", index=False)
    print(f"\nSaved: {RESULTS_DIR / '03_embedding_comparison.csv'}")

    # Grouped bar chart
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("Embedding Model Comparison - All RAG Architectures",
                 fontsize=14, fontweight="bold")
    plot_metrics = ["Accuracy", "RAGAS_Faithfulness", "RAGAS_Hallucination_Rate",
                    "Recall@5", "MRR", "F1"]
    for ax, metric in zip(axes.flat, plot_metrics):
        if metric in comp_df.columns:
            pivot = comp_df.pivot(index="RAG", columns="Embedding", values=metric)
            pivot.plot(kind="bar", ax=ax, edgecolor="black")
            ax.set_title(metric); ax.set_ylabel("Score"); ax.set_ylim(0, 1.05)
            ax.tick_params(axis="x", rotation=45); ax.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / "plots" / "03_embedding_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print("\nAll experiments complete!")
else:
    print("No results.")